In [0]:
# imports and read bronze
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

df = spark.table("ipl_catalog.bronze.matches")
print(f"Bronze matches row count: {df.count()}")

In [0]:
#standardise team names
#Old IPL teams got renamed over the years. Without this fix, "Delhi Daredevils" and "Delhi Capitals" look like 2 #different teams in your Gold analytics.
team_name_map = {
    "Delhi Daredevils": "Delhi Capitals",
    "Deccan Chargers": "Sunrisers Hyderabad",
    "Kings XI Punjab": "Punjab Kings",
    "Rising Pune Supergiant": "Rising Pune Supergiants",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru"
}

def standardise_team(col_name):
    mapping_expr = F.create_map(
        [F.lit(x) for pair in team_name_map.items() for x in pair]
    )
    return F.coalesce(mapping_expr[F.col(col_name)], F.col(col_name))

df = df.withColumn("team1", standardise_team("team1")) \
       .withColumn("team2", standardise_team("team2")) \
       .withColumn("toss_winner", standardise_team("toss_winner")) \
       .withColumn("winner", standardise_team("winner"))

In [0]:
#handle nulls in winner and result columns
# winner is null when result is "no result" or in rare tie/super_over cases. 
# Don't drop these rows — they are real matches, just flag them clearly.
df = df.withColumn(
    "winner",
    F.when(F.col("winner").isNull(), "No Result")
     .otherwise(F.col("winner"))
)

df = df.withColumn(
    "player_of_match",
    F.when(F.col("player_of_match").isNull(), "N/A")
     .otherwise(F.col("player_of_match"))
)

In [0]:
#fix data types — string to numeric
#result_margin, target_runs, target_overs came in as string from CSV. Cast them properly so you can do math and aggregations on them later in Gold.
# df = df.withColumn("result_margin", F.col("result_margin").cast(IntegerType())) \
#        .withColumn("target_runs", F.col("target_runs").cast(IntegerType())) \
#        .withColumn("target_overs", F.col("target_overs").cast("double"))



df = df.withColumn("result_margin", F.expr("try_cast(result_margin AS INT)")) \
       .withColumn("target_runs", F.expr("try_cast(target_runs AS INT)")) \
       .withColumn("target_overs", F.expr("try_cast(target_overs AS DOUBLE)"))

In [0]:
df.select("result_margin", "target_runs", "target_overs").show(10)

In [0]:
#add derived columns for analysis
#These flags make your Gold layer aggregations much simpler later. Build them once here.
df = df.withColumn("toss_winner_won_match", 
        F.when(F.col("toss_winner") == F.col("winner"), True).otherwise(False)) \
       .withColumn("is_super_over", 
        F.when(F.col("super_over") == "Y", True).otherwise(False)) \
       .withColumn("season_year",
        F.year(F.col("date")))

In [0]:
# write to silver
spark.sql("CREATE SCHEMA IF NOT EXISTS ipl_catalog.silver")

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ipl_catalog.silver.matches")

spark.sql("""
    ALTER TABLE ipl_catalog.silver.matches
    SET TBLPROPERTIES (
        'comment' = 'Cleaned IPL matches - standardised team names, fixed types, derived flags',
        'layer' = 'silver'
    )
""")


In [0]:
spark.table("ipl_catalog.silver.matches") \
    .select("team1", "team2", "winner", "result_margin", "season_year") \
    .show(5, truncate=False)

null_winners = spark.table("ipl_catalog.silver.matches") \
    .filter(F.col("winner").isNull()).count()
print(f"Null winners remaining: {null_winners}")